In [8]:
import requests
import pandas as pd
import json
import re

def get_yageo_dataframe():
    """
    YAGEO 월간 매출 데이터프레임 반환 (Playwright 불필요)
    """

    # Next.js 데이터 엔드포인트 시도
    possible_urls = [
        "https://www.yageo.com/_next/data/9AlMXHM01p4fCA3O0Z6iU/About/InvestorRelations.json",
        "https://www.yageo.com/api/investor-relations",
        "https://www.yageo.com/About/InvestorRelations"
    ]

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Accept': 'application/json, text/html, */*',
        'Accept-Language': 'en-US,en;q=0.9',
    }

    # 방법 1: Next.js build ID를 가진 JSON 엔드포인트
    print("Next.js 데이터 엔드포인트 시도 중...")

    for url in possible_urls:
        try:
            response = requests.get(url, headers=headers, timeout=30)

            if response.status_code == 200:
                print(f"✓ 연결 성공: {url}")

                # JSON 응답인 경우
                if 'application/json' in response.headers.get('Content-Type', ''):
                    data = response.json()
                    print("JSON 데이터 구조:")
                    print(json.dumps(data, indent=2)[:1000])
                    # 여기서 데이터 파싱 로직 추가
                    return parse_next_data(data)

                # HTML 응답인 경우 __NEXT_DATA__ 추출
                else:
                    html = response.text

                    # __NEXT_DATA__ 스크립트 찾기
                    match = re.search(r'<script id="__NEXT_DATA__"[^>]*>(.*?)</script>', html, re.DOTALL)

                    if match:
                        next_data = json.loads(match.group(1))
                        print("✓ __NEXT_DATA__ 발견")
                        return parse_next_data(next_data)

                    print("__NEXT_DATA__를 찾을 수 없습니다.")

        except Exception as e:
            print(f"  시도 실패: {url} - {e}")
            continue

    print("\n❌ 모든 방법 실패")
    print("\n대안: 브라우저 개발자 도구에서 확인")
    print("1. https://www.yageo.com/About/InvestorRelations 접속")
    print("2. F12 → Network 탭 → XHR/Fetch 필터")
    print("3. 페이지 새로고침")
    print("4. 월간 매출 데이터를 포함한 JSON 파일 찾기")
    print("5. Request URL 복사해서 알려주세요")

    return pd.DataFrame()


def parse_next_data(data):
    """
    Next.js 데이터 구조에서 월간 매출 정보 파싱
    """
    # 데이터 구조 탐색
    try:
        # Next.js 일반적인 구조
        if 'pageProps' in data:
            props = data['pageProps']
            print("pageProps 발견")
            print(json.dumps(props, indent=2)[:500])

        # 여기서 실제 데이터 위치에 따라 파싱 로직 작성
        # 예: props['monthlySales'], props['data'], etc.

        return pd.DataFrame()

    except Exception as e:
        print(f"파싱 오류: {e}")
        return pd.DataFrame()


# 임시 해결책: 수동 데이터 입력 함수
def create_yageo_dataframe_manual():
    """
    개발자 도구에서 확인한 데이터를 수동으로 입력

    사용법:
    1. 브라우저에서 F12 → Console
    2. 아래 JavaScript 실행:

    let data = [];
    document.querySelectorAll('table tbody tr').forEach(row => {
        let cells = row.querySelectorAll('td');
        if (cells.length >= 5) {
            data.push({
                month: cells[0].innerText,
                current: cells[1].innerText,
                previous: cells[2].innerText,
                mom: cells[3].innerText,
                yoy: cells[4].innerText
            });
        }
    });
    console.log(JSON.stringify(data));

    3. 출력된 JSON을 복사하여 아래에 붙여넣기
    """

    # 예시 데이터 (실제 데이터로 교체 필요)
    sample_data = [
        {"month": "January", "current": "52432", "previous": "41297", "mom": "-15.3", "yoy": "26.9"},
        {"month": "February", "current": "42607", "previous": "42432", "mom": "-18.7", "yoy": "0.4"},
        # ... 나머지 데이터
    ]

    # 파싱
    all_data = []

    month_mapping = {
        'january': 1, 'february': 2, 'march': 3, 'april': 4,
        'may': 5, 'june': 6, 'july': 7, 'august': 8,
        'september': 9, 'october': 10, 'november': 11, 'december': 12
    }

    for row in sample_data:
        month_num = month_mapping.get(row['month'].lower())
        if month_num:
            all_data.append({
                'year': 2024,  # 연도 수동 지정
                'month': month_num,
                'date': f"2024-{month_num:02d}",
                'current_sales': float(row['current'].replace(',', '')),
                'prev_sales': float(row['previous'].replace(',', '')),
                'mom': float(row['mom']),
                'yoy': float(row['yoy'])
            })

    return pd.DataFrame(all_data)


# if __name__ == "__main__":
#     print("YAGEO 데이터 수집 중...\n")
#
#     df = get_yageo_dataframe()
#
#     if df.empty:
#         print("\n" + "="*60)
#         print("브라우저에서 실제 API URL 확인 필요")
#         print("="*60)
#         print("\n아래 단계를 따라주세요:")
#         print("\n1. Chrome에서 https://www.yageo.com/About/InvestorRelations 접속")
#         print("2. F12 눌러서 개발자 도구 열기")
#         print("3. Network 탭 선택")
#         print("4. XHR 또는 Fetch 필터 선택")
#         print("5. 페이지 새로고침 (Ctrl+R)")
#         print("6. 'investor' 또는 'sales' 관련 요청 찾기")
#         print("7. 해당 요청의 URL을 복사해서 알려주세요")
#         print("\n예: https://www.yageo.com/api/v1/monthly-sales")

In [9]:
df = get_yageo_dataframe()

Next.js 데이터 엔드포인트 시도 중...
✓ 연결 성공: https://www.yageo.com/_next/data/9AlMXHM01p4fCA3O0Z6iU/About/InvestorRelations.json
__NEXT_DATA__를 찾을 수 없습니다.
✓ 연결 성공: https://www.yageo.com/api/investor-relations
__NEXT_DATA__를 찾을 수 없습니다.
✓ 연결 성공: https://www.yageo.com/About/InvestorRelations
__NEXT_DATA__를 찾을 수 없습니다.

❌ 모든 방법 실패

대안: 브라우저 개발자 도구에서 확인
1. https://www.yageo.com/About/InvestorRelations 접속
2. F12 → Network 탭 → XHR/Fetch 필터
3. 페이지 새로고침
4. 월간 매출 데이터를 포함한 JSON 파일 찾기
5. Request URL 복사해서 알려주세요


In [10]:
import pandas as pd
import json

def create_dataframe_from_json(json_string):
    """
    브라우저에서 추출한 JSON 문자열을 데이터프레임으로 변환

    Parameters:
    json_string: 브라우저 콘솔에서 복사한 JSON 문자열

    Returns:
    DataFrame: 정제된 월간 매출 데이터
    """
    # JSON 파싱
    data = json.loads(json_string)

    all_records = []

    month_mapping = {
        'january': 1, 'jan': 1,
        'february': 2, 'feb': 2,
        'march': 3, 'mar': 3,
        'april': 4, 'apr': 4,
        'may': 5,
        'june': 6, 'jun': 6,
        'july': 7, 'jul': 7,
        'august': 8, 'aug': 8,
        'september': 9, 'sep': 9,
        'october': 10, 'oct': 10,
        'november': 11, 'nov': 11,
        'december': 12, 'dec': 12
    }

    for row in data:
        month_text = row['month'].lower()

        # 월 번호 찾기
        month_num = None
        for month_name, num in month_mapping.items():
            if month_name in month_text:
                month_num = num
                break

        if not month_num:
            continue

        try:
            # 숫자 정제
            current_sales = row['current_sales'].replace(',', '').replace('$', '').strip()
            prev_sales = row['prev_sales'].replace(',', '').replace('$', '').strip()
            mom = row['mom'].replace('%', '').replace(',', '').strip()
            yoy = row['yoy'].replace('%', '').replace(',', '').strip()

            # 빈 값 처리
            if current_sales and current_sales not in ['-', 'N/A', '']:
                record = {
                    'year': int(row['year']),
                    'month': month_num,
                    'date': f"{row['year']}-{month_num:02d}",
                    'current_year': row['year'],
                    'current_sales': float(current_sales) if current_sales else None,
                    'prev_year': str(int(row['year']) - 1),
                    'prev_sales': float(prev_sales) if prev_sales and prev_sales not in ['-', 'N/A', ''] else None,
                    'mom': float(mom) if mom and mom not in ['-', 'N/A', '', '--'] else None,
                    'yoy': float(yoy) if yoy and yoy not in ['-', 'N/A', '', '--'] else None
                }
                all_records.append(record)

        except (ValueError, KeyError) as e:
            print(f"파싱 오류 ({row['month']}): {e}")
            continue

    # 데이터프레임 생성
    df = pd.DataFrame(all_records)

    # 정렬
    df = df.sort_values(['year', 'month']).reset_index(drop=True)

    # 중복 제거
    df = df.drop_duplicates(subset=['year', 'month'], keep='first')

    return df


# 사용 예시
if __name__ == "__main__":
    print("사용 방법:")
    print("1. 브라우저에서 F12 → Console 탭")
    print("2. 위의 JavaScript 코드 실행")
    print("3. 복사된 JSON 데이터를 아래 변수에 붙여넣기")
    print()

    # 여기에 브라우저에서 복사한 JSON 붙여넣기
    json_data = """
[
  {
    "year": "2024",
    "month": "January",
    "current_sales": "52,432",
    "prev_sales": "41,297",
    "mom": "-15.3",
    "yoy": "26.9"
  },
  {
    "year": "2024",
    "month": "February",
    "current_sales": "42,607",
    "prev_sales": "42,432",
    "mom": "-18.7",
    "yoy": "0.4"
  }
]
"""

    # 데이터프레임 생성
    try:
        df = create_dataframe_from_json(json_data)

        print(f"\n✓ {len(df)}개의 데이터 로드 완료\n")
        print("데이터 미리보기:")
        print(df.head(10))

        print("\n데이터 정보:")
        print(df.info())

        print("\n통계:")
        print(df[['current_sales', 'yoy']].describe())

    except Exception as e:
        print(f"오류: {e}")
        print("\nJSON 데이터를 확인해주세요.")

사용 방법:
1. 브라우저에서 F12 → Console 탭
2. 위의 JavaScript 코드 실행
3. 복사된 JSON 데이터를 아래 변수에 붙여넣기


✓ 2개의 데이터 로드 완료

데이터 미리보기:
   year  month     date current_year  current_sales prev_year  prev_sales  \
0  2024      1  2024-01         2024        52432.0      2023     41297.0   
1  2024      2  2024-02         2024        42607.0      2023     42432.0   

    mom   yoy  
0 -15.3  26.9  
1 -18.7   0.4  

데이터 정보:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   year           2 non-null      int64  
 1   month          2 non-null      int64  
 2   date           2 non-null      object 
 3   current_year   2 non-null      object 
 4   current_sales  2 non-null      float64
 5   prev_year      2 non-null      object 
 6   prev_sales     2 non-null      float64
 7   mom            2 non-null      float64
 8   yoy            2 non-null      float64
dtypes: float

In [11]:
df

,year,month,date,current_year,current_sales,prev_year,prev_sales,mom,yoy
0,2024,1,2024-01,2024,52432.0,2023,41297.0,-15.3,26.9
1,2024,2,2024-02,2024,42607.0,2023,42432.0,-18.7,0.4
